# Advanced Deep Learning Techniques (අතිශය වැදගත් සංකල්ප ගැඹුරින්)

අපි මේ වෙනකන් Loss Functions, Optimizers සහ Backpropagation වගේ මූලික දේවල් ඉගෙන ගත්තා. හැබැයි අපි ඇත්ත ලෝකයේ (production) Deep Learning Model එකක් හදද්දී, මේ න්‍යායාත්මක දේවල් වලට අමතරව තව ලොකු ප්‍රායෝගික ප්‍රශ්න කීපයකට මුහුණ දෙන්න වෙනවා.

ගොඩක් වෙලාවට beginners ලා හිර වෙන්නේ මේ ප්‍රශ්න වලදී. ඒ නිසා අපි මේ Notebook එකෙන් ඒ ප්‍රධාන ප්‍රශ්න 3ක් සහ ඒවා විසඳන ක්‍රම ගැන කිසිම කම්මැලිකමක් නැතුව, හරිම සරලව හැබැයි ගොඩක් ගැඹුරින් (deeply) කතා කරනවා:

1. **Exploding Gradient Problem** (Gradient එක පිපිරී යාම)
2. **Weight Initialization Techniques** (මුලින්ම Weights වලට අගයන් දෙන්නේ කොහොමද?)
3. **Dropout Layers** (Overfitting නවත්වන අපූරු උපක්‍රමය)

In [ ]:
import numpy as np
import tensorflow as tf

---
## 1. Exploding Gradient Problem (Gradient එක පිපිරී යාම)

අපි කලින් කතා කළා **Vanishing Gradient** (Gradient එක නැතිවී යාම) ගැන. ඒකෙදී වුණේ Chain Rule එකෙන් 1ට වඩා අඩු අගයන් ගොඩක් ගුණ වෙද්දී (උදා: 0.2 * 0.2 * 0.2 * 0.2...) ඒක බිංදුවටම කිට්ටු වෙන එක. එතකොට මුල් layers වලට ඉගෙනගන්න මුකුත් ඉතුරු වෙන්නේ නෑ.

**Exploding Gradient** කියන්නේ අන්න ඒකෙම අනිත් පැත්ත. මේක හරියට කටකතාවක් (rumor) වගේ. එක්කෙනෙක් ගිහින් අනිත් කෙනාට කියද්දී පොඩ්ඩක් ලුණු ඇඹුල් දාලා කියනවා, එයා ඊළඟ කෙනාට කියද්දී තවත් වැඩි කරලා කියනවා. අන්තිමට ඒ කතාව මුලින් අහපු දේට වඩා අතිවිශාල බොරුවක් වෙලා ඉවරයි නේද?

### 1.1 ගණිතමය පසුබිම:
හිතන්න ඔයාගේ Network එකේ Weights වල අගයන් 1ට වඩා ටිකක් වැඩියි කියලා (උදා: 1.5). Backpropagation වෙද්දී අපි දන්නවා Chain Rule එකෙන් මේ Weights ඔක්කොම එකින් එක ගුණ වේගෙන යනවා කියලා. 

අපි හිතමු Layers 100ක් තියෙනවා කියලා. එතකොට 1.5 කියන අගය 100 වතාවක් ගුණ වෙනවා (1.5^100). මේක අතිවිශාල අගයක්! (400,000,000,000,000,000 කට වඩා වැඩියි).

### 1.2 මේක වුණාම මොකද වෙන්නේ?
- Gradient එක (කන්දේ බෑවුම) අතිවිශාල වෙනවා. 
- අපි Weights update කරන්නේ: `New_Weight = Old_Weight - (Learning_Rate * Gradient)`
- Gradient එක බිලියන ගාණක් වුණාම, Weight එක වෙනස් වෙන්නෙත් බිලියන ගාණකින්. එතකොට Model එක කන්දෙන් පල්ලෙහාට යනව වෙනුවට, කන්දෙන් එහා පැත්තේ තියෙන වෙනමම රටකට විසි වෙලා යනවා වගේ වැඩක් වෙන්නේ.
- ඔයාගේ Loss එක එකපාරටම අහසට යනවා. අන්තිමට Python වල `NaN` (Not a Number) කියලා error එකක් එනවා. ඔයාගේ Model එක සම්පූර්ණයෙන්ම කඩාගෙන වැටෙනවා.

### 1.3 විසඳුම: Gradient Clipping
මේකට තියෙන ප්‍රධානම සහ සරලම විසඳුම තමයි **Gradient Clipping** කියන්නේ. ඒ කියන්නේ අපි කල්තියාම නීතියක් දානවා "Gradient එක කොච්චර ලොකු වුණත්, මේ සීමාවට වඩා යන්න බෑ" කියලා.

ක්‍රම දෙකක් තියෙනවා:
1. **Clip by Value:** කෙලින්ම අගය සීමා කරනවා. උදාහරණයක් විදිහට අපි සීමාව 1.0 කියලා දුන්නොත්, Gradient එක 50 ආවත් ඒක බලෙන් 1.0 කරනවා. -50 ආවොත් -1.0 කරනවා.
2. **Clip by Norm:** අගය කෙලින්ම කපලා දාන්නේ නැතුව, මුළු Gradient vector එකේම දිග (magnitude) එක නිශ්චිත අගයකට scale down කරනවා. මේක තමයි ගොඩක් වෙලාවට වඩා හොඳ ක්‍රමය, මොකද මේකෙන් ගමන් කරන දිශාව (direction) වෙනස් වෙන්නේ නෑ.

In [ ]:
# Keras වලදී Gradient Clipping කරන විදිහ

# 1. Clip by Value (-1.0 ත් 1.0 ත් අතරට සීමා කිරීම)
opt_value = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=1.0)

# 2. Clip by Norm (මුළු Vector එකේ දිග 1.0 ට සීමා කිරීම - වඩාත් සුදුසු ක්‍රමය)
opt_norm = tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0)

print("Gradient Clipping successfully configured! No more NaN errors!")

---
## 2. Weight Initialization Techniques (Weights මුලින්ම සකස් කිරීම)

Network එකක් train කරන්න පටන් ගනිද්දී, අපේ Weights වලට මොනවා හරි මුල් අගයන් (Initial values) දෙන්න එපැයි. මේ දෙන අගයන් හරියට දුන්නේ නැත්නම්, ඔයා මොන Optimizer එක දැම්මත් වැඩක් නෑ, Model එක train වෙන්නේ නෑ.

අපි බලමු මිනිස්සු මුලින් කරපු වැරදි මොනවද සහ දැන් තියෙන හොඳම ක්‍රම මොනවද කියලා.

### 2.1 Zero Initialization (ඔක්කොටම බිංදුව දීම - ලොකුම වැරැද්ද)
ඔක්කොම weights වලට මුලින් 0 දුන්නොත් මොකද වෙන්නේ? 
හිතන්න ඔෆිස් එකක වැඩ කරන සේවකයෝ 100ක් ඉන්නවා. හැබැයි මේ 100 දෙනාටම ලැබෙන්නේ එකම instruction එකක්. ඒ 100 දෙනාම කරන්නේ එකම වැඩේ. එතකොට සේවකයෝ 100ක් හිටියත්, ඇත්තටම වැඩ කරන්නේ එක සේවකයෙක් වගේ නේද?

Neural Network එකෙත් වෙන්නේ ඕකමයි. ඔක්කොම Weights බිංදුව වුණාම, හැම Neuron එකකටම ලැබෙන්නේ එකම input එකක්. Backpropagation වෙද්දී හැම Neuron එකකම Gradient එකත් එකම වෙනවා. ඒ නිසා ඔක්කොම Neurons එකම විදිහට update වෙනවා. මේකට කියන්නේ **Symmetry Problem** කියලා. Neuron 1000ක් තිබ්බත් ඒ ඔක්කොම එකම විදිහට හිතන්න ගත්තාම අපිට සංකීර්ණ දේවල් ඉගෙනගන්න බැරි වෙනවා.

### 2.2 Random Initialization (අහඹු අගයන් දීම - ප්‍රමාණවත් නෑ)
Symmetry ප්‍රශ්නය විසඳන්න අපි Weights වලට අහඹු අගයන් (Random values) දෙනවා. හැබැයි මෙතනත් ලොකු අවුලක් තියෙනවා:
- ගොඩක් **පොඩි අහඹු අගයන්** දුන්නොත්: Layers හරහා යද්දී signal එක එන්න එන්නම පොඩි වෙලා Vanishing Gradient ප්‍රශ්නය එනවා.
- ගොඩක් **ලොකු අහඹු අගයන්** දුන්නොත්: Layers හරහා යද්දී signal එක එන්න එන්නම ලොකු වෙලා Exploding Gradient ප්‍රශ්නය එනවා.

ඒ නිසා අපිට ඕනේ මේ දෙකම නොවෙන, හරියටම සමතුලිත (balanced) පරාසයක අගයන් ටිකක් දෙන ක්‍රමයක්.

### 2.3 Xavier / Glorot Initialization (Sigmoid/Tanh සඳහා)
Xavier Glorot කියන පර්යේෂකයා මේකට නියම ගණිතමය විසඳුමක් ගෙනාවා. එයා කිව්වා "අපි input එකේ තියෙන Variance එක (විසිරුම), output එකේදීත් ඒ විදිහටම තියාගන්න ඕනේ" කියලා.

මේක කරන්නේ කොහොමද? අහඹු අගයන් ගනිද්දී ඒකේ පරාසය (variance එක) තීරණය කරන්නේ කලින් Layer එකේ තියෙන Neurons ගාණ (fan-in) සහ ඊළඟ Layer එකේ Neurons ගාණ (fan-out) මත පදනම් වෙලයි.
**සූත්‍රය:** `Variance = 2 / (fan_in + fan_out)`

මේකෙන් අගයන් ගොඩක් ලොකු වෙන එකත්, ගොඩක් පොඩි වෙන එකත් සමතුලිත කරනවා.
**වැදගත්ම දේ:** මේක හරියටම වැඩ කරන්නේ Activation function එක විදිහට **Sigmoid** හෝ **Tanh** පාවිච්චි කරනවා නම් විතරයි.

### 2.4 He Initialization (ReLU සඳහා)
Deep Learning වලදී ගොඩක්ම පාවිච්චි කරන්නේ Sigmoid නෙවෙයි, ReLU නේ. හැබැයි Xavier Initialization එක ReLU එකට හරියන්නේ නෑ. ඇයි ඒ?
ReLU එකෙන් කරන්නේ සෘණ (negative) අගයන් ඔක්කොම බිංදුව කරන එක. ඒ කියන්නේ අපේ දත්ත වලින් හරි අඩක්ම (50% ක්ම) මරලා දානවා! එතකොට Xavier එකෙන් ආපු Variance එක එකපාරටම භාගයක් වෙනවා.

Kaiming He කියන පර්යේෂකයා මේකට විසඳුමක් ගෙනාවා. එයා කිව්වා "ReLU එකෙන් 50%ක් මැරෙනවා නම්, අපි මුලින් Weights හදද්දී Variance එක දෙගුණයක් (x2) කරමු" කියලා.
**සූත්‍රය:** `Variance = 2 / fan_in`

**රන් රීතිය (Golden Rule):** 
ඔයාගේ Hidden layers වලට **ReLU** (හෝ Leaky ReLU) පාවිච්චි කරනවා නම්, හැමවෙලේම පාවිච්චි කරන්න ඕනේ **He Initialization** එක තමයි.

In [ ]:
# Keras වල Initialization පාවිච්චි කරන විදිහ
from tensorflow.keras.layers import Dense

# Rule 1: ReLU පාවිච්චි කරන නිසා He Initialization (he_normal) පාවිච්චි කරනවා
layer_relu = Dense(units=64, activation='relu', kernel_initializer='he_normal')

# Rule 2: Tanh/Sigmoid පාවිච්චි කරන නිසා Glorot/Xavier Initialization (glorot_normal) පාවිච්චි කරනවා
layer_tanh = Dense(units=64, activation='tanh', kernel_initializer='glorot_normal')

print("Proper weight initialization avoids Vanishing and Exploding gradients from the start!")

---
## 3. Dropout Layers (අරුම පුදුම Regularization ක්‍රමය)

Deep Learning Models ගොඩක් බලවත්. එයාලට parameters මිලියන ගාණක් තියෙන්න පුළුවන්. මේ බලවත්කම නිසාම එන ලොකුම ප්‍රශ්නය තමයි, එයාලා Training Data එක තේරුම් ගන්නවා වෙනුවට ඒක **කටපාඩම්** කරන එක. මේකට කියන්නේ **Overfitting** කියලා. එතකොට අලුත් දත්තයක් (Test data) දුන්නාම model එකට ඒකට හරියට උත්තර දෙන්න බෑ (Generalize වෙන්නේ නෑ).

මේක නවත්වන්න පාවිච්චි කරන ප්‍රධානම Regularization ක්‍රමයක් තමයි **Dropout** කියන්නේ. මේක Geoff Hinton (Godfather of AI) ගේ අදහසක්.

### 3.1 Dropout වැඩ කරන්නේ කොහොමද? (The Gym Analogy)
හිතන්න ඔයාගේ කණ්ඩායමේ ක්‍රීඩකයෝ 10ක් ඉන්නවා කියලා. මේකෙන් එක්කෙනෙක් ගොඩක් දක්ෂයි (Dominant Neuron). හැමදාම තරඟ වලදී මේ දක්ෂ කෙනා තමයි ඔක්කොම බර අදින්නේ. අනිත් 9 දෙනා නිකම් බලන් ඉන්නවා. දවසක් මේ දක්ෂ කෙනාට ලෙඩ වුණොත් මුළු කණ්ඩායමම පරදිනවා නේද?

Dropout වලින් කරන්නේ හැම පුහුණුවීමක් (Training step) වෙලාවෙදිම අහඹු විදිහට (Randomly) ක්‍රීඩකයෝ කීප දෙනෙක්ව නිදි කරවනවා (Drop කරනවා).
උදාහරණයක් විදිහට Dropout rate එක `0.2` (20%) නම්, හැම training step එකේදීම අපි අහඹු ලෙස Neurons 100කින් 20ක output එක තාවකාලිකව බිංදුව (0) කරනවා.

### 3.2 මේකෙන් ලැබෙන අතිවිශාල වාසිය මොකක්ද?
දැන් අර දක්ෂ කෙනාට හැමදාම සෙල්ලම් කරන්න බෑ. එයා random දවස් වලට ගෙදර. එතකොට අනිත් 9 දෙනාට අකමැත්තෙන් වුණත් තනිවම වැඩ කරන්න පුරුදු වෙන්න සිද්ධ වෙනවා. 
Neural Network එකෙත් වෙන්නේ ඒකමයි! Dropout නිසා Network එකේ කිසිම Neuron එකකට තවත් එක Neuron එකක් මත (Co-adaptation) යැපෙන්න බැරි වෙනවා. ඒ නිසා හැම Neuron එකක්ම ස්වාධීනව හොඳ features ඉගෙනගන්න පෙළඹෙනවා. මේකෙන් Overfitting සම්පූර්ණයෙන්ම වගේ නැති වෙලා යනවා.

### 3.3 Testing / Prediction වෙලාවට මොකද වෙන්නේ? (Inference)
Training ඉවර වුණාට පස්සේ අපි ඇත්තටම Prediction ගන්නකොට කිසිම Neuron එකක් Drop කරන්නේ නෑ. දැන් කණ්ඩායමේ ඔක්කොම 10 දෙනාම එකට සෙල්ලම් කරනවා (ඒකනේ අපිට ඕනේ!).

හැබැයි පොඩි ප්‍රශ්නයක් තියෙනවා: Training වෙලාවේ වැඩ කළේ 8 දෙනයි. දැන් 10ම වැඩ කරන නිසා Output එකේ එකතුව (Sum) ගොඩක් ලොකු වෙන්න පුළුවන්. ඒක balance කරන්න Frameworks (වගේ TensorFlow) ස්වයංක්‍රීයව Weights වල අගයන් Dropout Rate එකට අනුපාතිකව අඩු කරනවා (Scale down කරනවා). ඒ නිසා අපි ඒ ගැන වධ වෙන්න ඕනේ නෑ.

In [ ]:
# Keras වල Dropout පාවිච්චි කරන විදිහ
from tensorflow.keras.layers import Dropout

model = tf.keras.Sequential([
    Dense(128, activation='relu', kernel_initializer='he_normal'),
    
    # 30% ක Neurons අහඹු ලෙස drop කරනවා (Training වෙලාවට විතරයි!)
    Dropout(0.3),
    
    Dense(64, activation='relu', kernel_initializer='he_normal'),
    
    # තවත් 20% ක් drop කරනවා
    Dropout(0.2),
    
    Dense(1, activation='sigmoid')
])

print("Model with Dropout is ready! This acts as an ensemble of many smaller networks.")